# Assignment 1

In this assignment, you'll be working with messy medical data and using regex to extract relevant infromation from the data. 

Each line of the `dates.txt` file corresponds to a medical note. Each note has a date that needs to be extracted, but each date is encoded in one of many formats.

The goal of this assignment is to correctly identify all of the different date variants encoded in this dataset and to properly normalize and sort the dates. 

Here is a list of some of the variants you might encounter in this dataset:
* 04/20/2009; 04/20/09; 4/20/09; 4/3/09
* Mar-20-2009; Mar 20, 2009; March 20, 2009;  Mar. 20, 2009; Mar 20 2009;
* 20 Mar 2009; 20 March 2009; 20 Mar. 2009; 20 March, 2009
* Mar 20th, 2009; Mar 21st, 2009; Mar 22nd, 2009
* Feb 2009; Sep 2009; Oct 2010
* 6/2008; 12/2009
* 2009; 2010

Once you have extracted these date patterns from the text, the next step is to sort them in ascending chronological order accoring to the following rules:
* Assume all dates in xx/xx/xx format are mm/dd/yy
* Assume all dates where year is encoded in only two digits are years from the 1900's (e.g. 1/5/89 is January 5th, 1989)
* If the day is missing (e.g. 9/2009), assume it is the first day of the month (e.g. September 1, 2009).
* If the month is missing (e.g. 2010), assume it is the first of January of that year (e.g. January 1, 2010).
* Watch out for potential typos as this is a raw, real-life derived dataset.

With these rules in mind, find the correct date in each note and return a pandas Series in chronological order of the original Series' indices. **This Series should be sorted by a tie-break sort in the format of ("extracted date", "original row number").**

For example if the original series was this:

    0    1999
    1    2010
    2    1978
    3    2015
    4    1985

Your function should return this:

    0    2
    1    4
    2    0
    3    1
    4    3

Your score will be calculated using [Kendall's tau](https://en.wikipedia.org/wiki/Kendall_rank_correlation_coefficient), a correlation measure for ordinal data.

*This function should return a Series of length 500 and dtype int.*

In [1]:
import pandas as pd

doc = []
with open('assets/dates.txt') as file:
    for line in file:
        doc.append(line)

df = pd.Series(doc)
df.head(10)

0         03/25/93 Total time of visit (in minutes):\n
1                       6/18/85 Primary Care Doctor:\n
2    sshe plans to move as of 7/8/71 In-Home Servic...
3                7 on 9/27/75 Audit C Score Current:\n
4    2/6/96 sleep studyPain Treatment Pain Level (N...
5                    .Per 7/06/79 Movement D/O note:\n
6    4, 5/18/78 Patient's thoughts about current su...
7    10/24/89 CPT Code: 90801 - Psychiatric Diagnos...
8                         3/7/86 SOS-10 Total Score:\n
9             (4/10/71)Score-1Audit C Score Current:\n
dtype: str

In [2]:
def date_sorter():
    
    order = None
    # YOUR CODE HERE
    from datetime import datetime
    import re
    months_map = {
        'jan': 1, 'january': 1,
        'feb': 2, 'february': 2,
        'mar': 3, 'march': 3,
        'apr': 4, 'april': 4,
        'may': 5,
        'jun': 6, 'june': 6,
        'jul': 7, 'july': 7,
        'aug': 8, 'august': 8,
        'sep': 9, 'sept': 9, 'september': 9,
        'oct': 10, 'october': 10,
        'nov': 11, 'november': 11,
        'dec': 12, 'december': 12,
    }
 
    MONTH_RE = (
        r'Jan\.?|Feb\.?|Mar\.?|Apr\.?|May\.?|Jun\.?|Jul\.?|Aug\.?|'
        r'Sep\.?|Sept\.?|Oct\.?|Nov\.?|Dec\.?|'
        r'January|February|March|April|June|July|August|'
        r'September|October|November|December'
    )
 
    parsed_dates = []
 
    for i, text in enumerate(df):
        date = None
 
        # ── Pattern 1: mm/dd/yyyy  or  mm/dd/yy  (also handles typo like 011/14/83)
        m = re.search(r'\b0*(\d{1,2})[/-](\d{1,2})[/-](\d{2,4})\b', text)
        if m:
            mo, day, yr = int(m.group(1)), int(m.group(2)), int(m.group(3))
            if yr < 100:
                yr += 1900
            try:
                date = datetime(yr, mo, day)
            except ValueError:
                pass
 
        # ── Pattern 2: Month dd, yyyy  /  Mon. dd yyyy  /  Mar 20th, 2009 ────
        if date is None:
            m = re.search(
                rf'\b({MONTH_RE})[\s\-](\d{{1,2}})(?:st|nd|rd|th)?[,\s]+(\d{{4}})\b',
                text, re.IGNORECASE
            )
            if m:
                mo = months_map[m.group(1).rstrip('.').lower()]
                day, yr = int(m.group(2)), int(m.group(3))
                try:
                    date = datetime(yr, mo, day)
                except ValueError:
                    pass
 
        # ── Pattern 3: dd Month yyyy  /  20 March, 2009 ───────────────────────
        if date is None:
            m = re.search(
                rf'\b(\d{{1,2}})[\s\-]({MONTH_RE})[,\s]+(\d{{4}})\b',
                text, re.IGNORECASE
            )
            if m:
                day = int(m.group(1))
                mo = months_map[m.group(2).rstrip('.').lower()]
                yr = int(m.group(3))
                try:
                    date = datetime(yr, mo, day)
                except ValueError:
                    pass
 
        # ── Pattern 4: Month yyyy  (no day — assume day=1) ───────────────────
        if date is None:
            m = re.search(
                rf'\b({MONTH_RE})[,\s]+(\d{{4}})\b',
                text, re.IGNORECASE
            )
            if m:
                mo = months_map[m.group(1).rstrip('.').lower()]
                yr = int(m.group(2))
                date = datetime(yr, mo, 1)
 
        # ── Pattern 5: m/yyyy  (no day — assume day=1) ───────────────────────
        if date is None:
            m = re.search(r'\b(\d{1,2})/(\d{4})\b', text)
            if m:
                mo, yr = int(m.group(1)), int(m.group(2))
                if 1 <= mo <= 12:
                    date = datetime(yr, mo, 1)
 
        # ── Pattern 6: yyyy only  (also handles letter-prefix typos like y1983)
        if date is None:
            m = re.search(r'(?<!\d)((?:19|20)\d{2})(?!\d)', text)
            if m:
                yr = int(m.group(1))
                date = datetime(yr, 1, 1)
 
        parsed_dates.append((date, i))
 
    # Sort chronologically; tie-break by original row index
    parsed_dates.sort(key=lambda x: (x[0] or datetime(9999, 1, 1), x[1]))
 
    order = pd.Series([idx for _, idx in parsed_dates], dtype=int)
       
    return order # Your answer here

In [3]:
date_sorter()

0        9
1       84
2        2
3       53
4       28
      ... 
495    427
496    141
497    186
498    161
499    413
Length: 500, dtype: int64

In [4]:
import re
import numpy as np
s_test = date_sorter()

def run_df_modified_check():
    """
    Check if df appears to be modified.
    """
    try:
        assert type(df) == pd.Series
        assert (df.index == pd.RangeIndex(start=0, stop=500, step=1)).all()
        assert (df.apply(type) == str).all()
        assert df.str.len().min() >= 6
        assert df.str[5].apply(ord).sum() == 38354
        print("Passed df modification check")
    except:
        print("Failed df modification check")

run_df_modified_check()

# check if running the code twice produces the same result
try:
    assert (date_sorter() == s_test).all()
    print("Passed repeatability check")
except:
    print("Failed repeatability check")

# check if the result has the expected index
try:
    # assert type(date_sorter().index) == pd.RangeIndex
    # assert (date_sorter().index == pd.RangeIndex(start=0, stop=500, step=1)).all()
    assert list(date_sorter().index) == list(range(500))
    print("Passed index check")
except:
    print("Failed index check")

# check the tie-break sort for a sample of records where some have the same date
# note that this only tests a sample and does not check the entire answer
try:
    test_indices = [335, 415, 323, 405, 370, 382, 303, 488, 283,
                    395, 318, 369, 493, 252, 314, 410, 490]
    answer_lkp = {original_index:answer_index for
                  answer_index, original_index in s_test.to_dict().items()}
    i_test = [answer_lkp[i] for i in test_indices]
    assert sorted(i_test) == i_test
    print("Passed secondary sort sample check")
except:
    print("Failed secondary sort sample check")

def run_v_check(s_test):
    """
    Check if the parsed dates appear to be correct and correctly sorted.
    The check works by producing some test checksums
    if you get for example a False entry in the agree column for
    index value 20 that would mean you have at least one incorrectly
    parsed or incorrectly sorted date in the **output** index
    range 20,21,...,29
    The results of the test are printed.
    Args:
    s_test: Series such as produced by date_sorter()
    Returns:
    None
    """
    try:
        v_check = pd.DataFrame({'correct':
        [6695, 14428, 16742, 9275, 12290, 14654, 9421, 10185, 11464, 16491,
         11797, 14036, 15459, 9412, 13069, 10400, 10498, 14322, 13274, 11001,
         11383, 11910, 10977, 9692, 10199, 10187, 15456, 13491, 9186, 13646,
         11142, 13724, 10994, 12905, 15968, 16648, 13966, 14607, 16932, 14622,
         17942, 18220, 17818, 18305, 19633, 12522, 13978, 18445, 20156, 14797],
        'learner':[
        (s_test.iloc[10*i:(i+1)*10].values * np.array(range(1,11))).sum() for i in range(50)]},
        index=range(0,500,10)).assign(agree=lambda x:x['correct']==x['learner'])
        print("Values checksums:")
        print(v_check)
        assert v_check['agree'].all()
        print("Passed values check")
    except:
        print("Failed values check")
    return

run_v_check(s_test)

Passed df modification check
Passed repeatability check
Passed index check
Failed secondary sort sample check
Values checksums:
     correct  learner  agree
0       6695     6695   True
10     14428    14428   True
20     16742    16742   True
30      9275     9275   True
40     12290    12290   True
50     14654    14654   True
60      9421     9221  False
70     10185    10410  False
80     11464    12104  False
90     16491    15870  False
100    11797    11797   True
110    14036    14036   True
120    15459    15459   True
130     9412     9412   True
140    13069    15730  False
150    10400     9787  False
160    10498    10498   True
170    14322    14322   True
180    13274    13274   True
190    11001    11001   True
200    11383    11444  False
210    11910    11910   True
220    10977    10977   True
230     9692     9719  False
240    10199    10160  False
250    10187    10187   True
260    15456    15456   True
270    13491    13491   True
280     9186     9186   True
29

In [5]:
import pandas as pd
import re

def date_sorterr():
    
    dates = []

    for i, text in enumerate(df):

        date = None

        # 1. mm/dd/yyyy or mm/dd/yy
        m = re.search(r'\b\d{1,2}/\d{1,2}/\d{2,4}\b', text)
        if m:
            date = m.group()

        # 2. mm-dd-yyyy
        if date is None:
            m = re.search(r'\b\d{1,2}-\d{1,2}-\d{2,4}\b', text)
            if m:
                date = m.group()

        # 3. Month dd, yyyy
        if date is None:
            m = re.search(r'\b(?:Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)[a-z]*\.?\s\d{1,2}(?:st|nd|rd|th)?,?\s\d{4}', text, re.I)
            if m:
                date = m.group()

        # 4. dd Month yyyy
        if date is None:
            m = re.search(r'\b\d{1,2}\s(?:Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)[a-z]*\.?,?\s\d{4}', text, re.I)
            if m:
                date = m.group()

        # 5. Month yyyy
        if date is None:
            m = re.search(r'\b(?:Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)[a-z]*\.?,?\s\d{4}', text, re.I)
            if m:
                date = m.group()

        # 6. mm/yyyy
        if date is None:
            m = re.search(r'\b\d{1,2}/\d{4}\b', text)
            if m:
                date = m.group()

        # 7. yyyy
        if date is None:
            m = re.search(r'\b\d{4}\b', text)
            if m:
                date = m.group()

        dates.append((i, date))


    # normalize dates
    parsed = []
    
    for i, d in dates:

        if d is None:
            parsed.append((i, pd.NaT))
            continue

        if re.match(r'^\d{4}$', d):
            d = 'Jan 1 ' + d

        elif re.match(r'^\d{1,2}/\d{4}$', d):
            d = '1/' + d

        parsed.append((i, pd.to_datetime(d, errors='coerce')))

   


    df_dates = pd.DataFrame(parsed, columns=['idx', 'date'])

    df_dates = df_dates.sort_values(['date','idx'])

    return pd.Series(df_dates['idx'].values)

In [6]:
date_sorterr()

0      474
1      153
2      129
3      225
4      171
      ... 
495    313
496    466
497    470
498    481
499    493
Length: 500, dtype: int64

In [7]:
import re
import numpy as np
s_test = date_sorterr()

def run_df_modified_check():
    """
    Check if df appears to be modified.
    """
    try:
        assert type(df) == pd.Series
        assert (df.index == pd.RangeIndex(start=0, stop=500, step=1)).all()
        assert (df.apply(type) == str).all()
        assert df.str.len().min() >= 6
        assert df.str[5].apply(ord).sum() == 38354
        print("Passed df modification check")
    except:
        print("Failed df modification check")

run_df_modified_check()

# check if running the code twice produces the same result
try:
    assert (date_sorter() == s_test).all()
    print("Passed repeatability check")
except:
    print("Failed repeatability check")

# check if the result has the expected index
try:
    # assert type(date_sorter().index) == pd.RangeIndex
    # assert (date_sorter().index == pd.RangeIndex(start=0, stop=500, step=1)).all()
    assert list(date_sorter().index) == list(range(500))
    print("Passed index check")
except:
    print("Failed index check")

# check the tie-break sort for a sample of records where some have the same date
# note that this only tests a sample and does not check the entire answer
try:
    test_indices = [335, 415, 323, 405, 370, 382, 303, 488, 283,
                    395, 318, 369, 493, 252, 314, 410, 490]
    answer_lkp = {original_index:answer_index for
                  answer_index, original_index in s_test.to_dict().items()}
    i_test = [answer_lkp[i] for i in test_indices]
    assert sorted(i_test) == i_test
    print("Passed secondary sort sample check")
except:
    print("Failed secondary sort sample check")

def run_v_check(s_test):
    """
    Check if the parsed dates appear to be correct and correctly sorted.
    The check works by producing some test checksums
    if you get for example a False entry in the agree column for
    index value 20 that would mean you have at least one incorrectly
    parsed or incorrectly sorted date in the **output** index
    range 20,21,...,29
    The results of the test are printed.
    Args:
    s_test: Series such as produced by date_sorter()
    Returns:
    None
    """
    try:
        v_check = pd.DataFrame({'correct':
        [6695, 14428, 16742, 9275, 12290, 14654, 9421, 10185, 11464, 16491,
         11797, 14036, 15459, 9412, 13069, 10400, 10498, 14322, 13274, 11001,
         11383, 11910, 10977, 9692, 10199, 10187, 15456, 13491, 9186, 13646,
         11142, 13724, 10994, 12905, 15968, 16648, 13966, 14607, 16932, 14622,
         17942, 18220, 17818, 18305, 19633, 12522, 13978, 18445, 20156, 14797],
        'learner':[
        (s_test.iloc[10*i:(i+1)*10].values * np.array(range(1,11))).sum() for i in range(50)]},
        index=range(0,500,10)).assign(agree=lambda x:x['correct']==x['learner'])
        print("Values checksums:")
        print(v_check)
        assert v_check['agree'].all()
        print("Passed values check")
    except:
        print("Failed values check")
    return

run_v_check(s_test)

Passed df modification check
Failed repeatability check
Passed index check
Failed secondary sort sample check
Values checksums:
     correct  learner  agree
0       6695    18655  False
10     14428    17674  False
20     16742    10825  False
30      9275    17838  False
40     12290     9221  False
50     14654    10300  False
60      9421    11952  False
70     10185    14304  False
80     11464    14190  False
90     16491    14080  False
100    11797    14305  False
110    14036     7527  False
120    15459    12236  False
130     9412    14971  False
140    13069     9215  False
150    10400    15735  False
160    10498     8612  False
170    14322     6320  False
180    13274    14690  False
190    11001    12574  False
200    11383    12277  False
210    11910    11472  False
220    10977     9173  False
230     9692    10235  False
240    10199    12973  False
250    10187    18804  False
260    15456    14687  False
270    13491     9174  False
280     9186    10574  False
29